# Benchmark 2 of 3: run models on the benchmark

Every model walks the same questions, and sees the same numbered options at every step. On WebQSP and CWQ, STOP is not
offered while the walk stands only on unnamed Freebase records (shown by their ID, such as `m.0j4jq57`). These records
link other entities together, so the walk moves on to the named entities behind them.

You can run two kinds of model:

- **Decision models**, which return a probability for every option, for example `liquid/d1-ultra`,
  `inception/mercury-decide` or `typesafe/jev-1.14`.
- **LLMs**, which are any chat model on OpenRouter, for example GPT-5.6 Luna, Claude or DeepSeek. An LLM answers with a
  small structured (Pydantic) answer that must name one of the offered options. Its answer is read directly, without
  any text parsing.

You need `OPENROUTER_API_KEY` in the `.env` file. A run of 50 questions costs about $0.01 to $0.03 with a decision
model, and about $0.07 with GPT-5.6 Luna.

## Settings

- `RUN_NAME`: leave it as `None` to get the next number (`run-001`, `run-002`, ...). Use the same name to continue a
  run that stopped, or to add models to it.
- `MODELS`: any mix of decision models and LLMs. Model names are on https://openrouter.ai/models.
- `LLM(..., reasoning=True)` lets a reasoning model think before it answers. It costs more, and is off by default.
- `BUDGET_USD`: each model stops once it has spent this much.

In [ ]:
BENCHMARK = "webqsp"     # the benchmark you loaded in 1_load_benchmark.ipynb
RUN_NAME = None
BUDGET_USD = 1.00
CONCURRENCY = 8          # questions walked at the same time

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import utils/ from the repository root

import json

import pandas as pd

from utils import DecisionModel, LLM, answer_model, display, graph, load_results, models, open_run, run_models, walk
from utils.settings import DATA

MODELS = [
    DecisionModel("liquid/d1-ultra"),
    LLM("openai/gpt-5.6-luna"),
    # DecisionModel("inception/mercury-decide"),
    # LLM("anthropic/claude-sonnet-4.5"),
    # LLM("deepseek/deepseek-chat-v3.1"),
    # LLM("openai/gpt-5.6-luna", reasoning=True),
]

loaded = json.loads((DATA / "loaded" / f"{BENCHMARK}.json").read_text())
questions = loaded["questions"]
print(f"{len(questions)} {BENCHMARK} questions, loaded by 1_load_benchmark.ipynb")

## What every model sees

This is the first step of the first question: the state (the question, where the walk stands, the path so far) and
the numbered options. Option 0 is always STOP, which means "the entities I'm standing on are the answer".

In [ ]:
q = questions[0]
driver = graph.connect_async()
rows = await graph.neighbours(driver, BENCHMARK, q, q["start"])
options = graph.build_options(rows, q["start"], BENCHMARK, q)
request = walk.build_request({"question": q["question"], "current_entities": q["start"], "path_so_far": [], "hop": 0}, options)
print(models.render_text(request)[:2500])

## The answer an LLM must give

The LLM's answer is a Pydantic model with one field, `option`, which must be one of the option numbers offered at this
step. Its JSON schema goes to the model as a strict response format, so the model can only reply with a valid option.
The reply is checked with the same Pydantic model.

In [ ]:
Answer = answer_model(tuple(request["questions"]["next_step"]["criteria"]))
schema = Answer.model_json_schema()
schema["properties"]["option"]["enum"] = schema["properties"]["option"]["enum"][:6] + ["..."]  # shortened for display
print(json.dumps(schema, indent=2))

print("\nA valid reply:  ", Answer.model_validate_json('{"option": "1"}'))
try:
    Answer.model_validate_json('{"option": "9999"}')
except Exception as error:
    print("An invalid reply is rejected:", type(error).__name__)

## Check the models

Each LLM must support structured output on OpenRouter. This also shows its price.

In [ ]:
async with models.client() as http:
    checks = [await m.check(http) for m in MODELS]
pd.DataFrame([{"model": c["slug"], "kind": c["kind"], "reasoning": c.get("reasoning", ""),
               "$ per million input tokens": (c.get("prices_usd") or {}).get("input_per_million", "see OpenRouter")}
              for c in checks])

## Run

A progress bar shows each model's questions and dollars spent. You can stop and rerun this cell; it continues where it stopped.

In [ ]:
run = open_run(BENCHMARK, questions, name=RUN_NAME)
print(f"Run: {run.name}  (results in benchmark/runs/{run.name}/{BENCHMARK}/)")
await run_models(run, MODELS, concurrency=CONCURRENCY, budget_usd=BUDGET_USD)
await driver.close()

## A first look

The full comparison is in `3_compare_results.ipynb`.

In [ ]:
display.summary(load_results(BENCHMARK, [run.name]))